In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("train.txt",sep=";",header=None, names=['text','emotion'])

In [4]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [5]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [6]:
unique_emotion = df['emotion'].unique()

In [7]:
unique_emotion

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [8]:
emotion_numbers = {}

In [9]:
i = 0
for emo in unique_emotion:
  emotion_numbers[emo]=i
  i+=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [10]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


1) **lowercasing**

In [11]:
df['text'] = df['text'].apply(lambda x: x.lower())

2) **Removing Punctuation**

In [12]:
import string
def remove_punc(text):
  return text.translate(str.maketrans('','',string.punctuation))


In [13]:
df['text'] = df['text'].apply(remove_punc)

In [14]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


3) **Removing Numbers**

In [15]:
def remove_numbers(text):
  new = ""
  for i in text:
    if not i.isdigit():
      new = new+i
  return new

  df['text']= df['text'].apply(remove_numbers)

In [16]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


4) **Removing Emojis and Special Characters**

In [17]:
def remove_emojis(text):
  new =""
  for i in text:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emojis)

4) **Removing Stopwords**

In [18]:
import nltk

In [19]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [20]:
nltk.download('punkt') # Used for tokenization
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/bipan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /home/bipan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [21]:
stop_words = set(stopwords.words('english'))

In [22]:
len(stop_words)

198

In [23]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [24]:
# Tokenization

In [25]:
def remove(text):
  words = text.split() #You can use words_tokenize(text) here instead of using split but i got problem here so.
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)


  return ' '.join(cleaned)

In [26]:
df['text'] = df['text'].apply(remove)

In [27]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [28]:
from sklearn.model_selection import train_test_split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [30]:
X_train

676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
                               ...                        
13418    love leave reader feeling confused slightly de...
5390                                         feel delicate
860                          starting feel little stressed
15795             feel stressed tired worn shape neglected
7270         feel someone rude wrongly done something lose
Name: text, Length: 12800, dtype: object

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

**Using Bag of Words**

In [32]:
bow_vectorizer = CountVectorizer()


In [33]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

In [34]:
X_test_bow = bow_vectorizer.transform(X_test)

In [35]:
from sklearn.naive_bayes import MultinomialNB
#GaussianNB is for Normal Distribution one,
#but we use MultinomialNB for classification task where features or input
# data are discrete such as word counts or frequencies in text classification

In [36]:
from sklearn.metrics import accuracy_score

In [37]:
NB_model = MultinomialNB()

In [38]:
NB_model.fit(X_train_bow,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [39]:
prediction_NB = NB_model.predict(X_test_bow)

In [40]:
accuracy_using_bow = accuracy_score(y_test,prediction_NB)

In [41]:
accuracy_using_bow

0.768125

**Using TF-IDF**

In [42]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [43]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


**Using Naive Bayes**

In [44]:
prediction_NB2 = nb2_model.predict(X_test_tfidf)

In [45]:
accuracy = accuracy_score(y_test,prediction_NB2)

In [46]:
accuracy

0.6609375

**TF-IDF using Logistic Regression**

In [47]:
from sklearn.linear_model import LogisticRegression

In [48]:
model_LR = LogisticRegression(max_iter = 1000)
# Default max_iter is 100 and if not converges you can increase this number

In [49]:
model_LR.fit(X_train_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [50]:
prediction_LR = model_LR.predict(X_test_tfidf)

In [51]:
accuracy = accuracy_score(y_test,prediction_LR)

In [52]:
accuracy

0.8628125

In [53]:
import pickle

In [54]:
# Save the TF-IDF vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# Save the trained Logistic Regression model
with open('model_LR.pkl', 'wb') as f:
    pickle.dump(model_LR, f)

# Save the emotion label mapping (so we can decode predictions back to text)
with open('emotion_numbers.pkl', 'wb') as f:
    pickle.dump(emotion_numbers, f)

print("Saved: tfidf_vectorizer.pkl, model_LR.pkl, emotion_numbers.pkl")

Saved: tfidf_vectorizer.pkl, model_LR.pkl, emotion_numbers.pkl
